In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown

data = pd.read_csv('HoC-GE2024-results-by-constituency.csv')
data.head()

,ONS ID,ONS region ID,Constituency name,County name,Region name,Country name,Constituency type,Declaration time,Member first name,Member surname,...,Green,SNP,PC,DUP,SF,SDLP,UUP,APNI,All other candidates,Of which other winner
0,W07000081,W92000004,Aberafan Maesteg,NaN,Wales,Wales,County,NaN,Stephen,Kinnock,...,1094,0,4719,0,0,0,0,0,801,0
1,S14000060,S92000003,Aberdeen North,NaN,Scotland,Scotland,Burgh,NaN,Kirsty,Blackman,...,1275,14533,0,0,0,0,0,0,1269,0
2,S14000061,S92000003,Aberdeen South,NaN,Scotland,Scotland,Burgh,NaN,Stephen,Flynn,...,1609,15213,0,0,0,0,0,0,648,0
3,S14000062,S92000003,Aberdeenshire North and Moray East,NaN,Scotland,Scotland,County,NaN,Seamus,Logan,...,0,13455,0,0,0,0,0,0,0,0
4,S14000063,S92000003,Airdrie and Shotts,NaN,Scotland,Scotland,County,NaN,Kenneth,Stevenson,...,0,11324,0,0,0,0,0,0,1079,0


In [14]:
data['Total Conservative Vote'] = data["Con"] + data['RUK']
vote_split = data.query("Lab < `Total Conservative Vote` and `First party` == 'Lab'")
vote_split.info()
vote_split.head()

<class 'pandas.DataFrame'>
Index: 144 entries, 5 to 643
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ONS ID                   144 non-null    str    
 1   ONS region ID            144 non-null    str    
 2   Constituency name        144 non-null    str    
 3   County name              0 non-null      float64
 4   Region name              144 non-null    str    
 5   Country name             144 non-null    str    
 6   Constituency type        144 non-null    str    
 7   Declaration time         0 non-null      float64
 8   Member first name        144 non-null    str    
 9   Member surname           144 non-null    str    
 10  Member gender            144 non-null    str    
 11  Result                   144 non-null    str    
 12  First party              144 non-null    str    
 13  Second party             144 non-null    str    
 14  Electorate               144 non-null    i

,ONS ID,ONS region ID,Constituency name,County name,Region name,Country name,Constituency type,Declaration time,Member first name,Member surname,...,SNP,PC,DUP,SF,SDLP,UUP,APNI,All other candidates,Of which other winner,Total Conservative Vote
5,E14001063,E12000008,Aldershot,NaN,South East,England,Borough,NaN,Alex,Baker,...,0,0,0,0,0,0,0,282,0,22291
8,E14001065,E12000002,Altrincham and Sale West,NaN,North West,England,Borough,NaN,Connor,Rand,...,0,0,0,0,0,0,0,643,0,21585
10,E14001066,E12000004,Amber Valley,NaN,East Midlands,England,County,NaN,Linsey,Farnsworth,...,0,0,0,0,0,0,0,0,0,22917
16,E14001069,E12000008,Ashford,NaN,South East,England,County,NaN,Sojan,Joseph,...,0,0,0,0,0,0,0,1289,0,23624
18,E14001071,E12000008,Aylesbury,NaN,South East,England,County,NaN,Laura,Kyrke-Smith,...,0,0,0,0,0,0,0,632,0,21197


In [ ]:
ALL = 'All (GB)'
countries = [ALL, 'England', 'Scotland', 'Wales']   # Northern Ireland left out

@interact(country=Dropdown(options=countries, description='Country'))
def labour_split_pie(country=ALL):
    seats = data[(data['First party'] == 'Lab') & (data['Country name'] != 'Northern Ireland')]
    if country != ALL:
        seats = seats[seats['Country name'] == country]

    n_split = (seats['Lab'] < seats['Total Conservative Vote']).sum()
    n_rest = len(seats) - n_split

    # Drop empty slices so a 0-seat label doesn't sit on top of the full slice
    slices = [(n, label, color) for n, label, color in [
        (n_split, 'Con + RUK > Lab', '#0087dc'),
        (n_rest, 'Lab ≥ Con + RUK', '#d50000'),
    ] if n > 0]
    sizes, labels, colors = zip(*slices)

    plt.figure(figsize=(5, 5))
    plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
    plt.title(f'Labour-won seats: {country} ({len(seats)} seats)')

    if n_split == 0:   # e.g. Scotland
        plt.annotate(f'Labour beat Con + RUK combined\nin all {n_rest} seats it won',
                     xy=(0.5, -0.05), xycoords='axes fraction', ha='center', va='top',
                     fontsize=11, fontweight='bold')
    plt.show()